In [0]:
%sql

-- Garantindo que o schema silver esteja inserido no catálogo medalhao
USE CATALOG medalhao;

-- Criando o schema silver
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
# Importando as funções do Spark 
from pyspark.sql import functions as F

# Carregando as tabelas da Silver que já passaram pelo tratamento, a 'fat_pedido_total' já tem o valor em Real e Dólar calculado
df_pedidos = spark.table("medalhao.silver.fat_pedido_total")
df_itens = spark.table("medalhao.silver.fat_itens_pedidos")
df_produtos = spark.table("medalhao.silver.dim_produtos")

# Criando a base de vendas através do Join das Fatos com a Dimensão de Produtos, o inner join garante que só analisaremos pedidos que possuem itens e produtos cadastrados
df_base_vendas = (
  df_pedidos.alias("pedidos")
  .join(df_itens.alias("itens"), "id_pedido", "inner")
  .join(df_produtos.alias("produtos"), "id_produto", "inner")
)

# Construindo a Camada Gold Comercial: Visão de Performance por Categoria e Tempo
df_gold_comercial = (
    df_base_vendas
    # Extraindo Ano e Mês da data do pedido para permitir filtros temporais no Dashboard
    .withColumn("ano_venda", F.year("data_pedido"))
    .withColumn("mes_venda", F.month("data_pedido"))
    
    # Agrupando por período e categoria para consolidar os números de negócio
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    
    .agg(
        # countDistinct: Garante a contagem real de vendas únicas (um pedido pode ter vários itens)
        F.countDistinct("pedidos.id_pedido").alias("total_pedidos"),
        
        # count: Soma a quantidade física de itens que saíram do estoque
        F.count("itens.id_produto").alias("qtd_itens_vendidos"),
        
        # sum: Totalizadores financeiros convertidos e arredondados
        F.sum("pedidos.valor_total_pago_brl").alias("receita_total_brl"),
        F.sum("pedidos.valor_total_pago_usd").alias("receita_total_usd")
    )
    
    # Cálculo de Ticket Médio para entender o gasto médio por cliente
    .withColumn("ticket_medio_brl", F.round(F.col("receita_total_brl") / F.col("total_pedidos"), 2))
    
    # Arredondamento final das colunas monetárias para evitar dízimas 
    .withColumn("receita_total_brl", F.round("receita_total_brl", 2))
    .withColumn("receita_total_usd", F.round("receita_total_usd", 2))
    
    # Ordenação lógica para que os dados fiquem organizados cronologicamente na tabela
    .orderBy("ano_venda", "mes_venda", "categoria_produto")
)
# Usamos o overwriteSchema para que o Job não quebre caso a estrutura da Silver mude, o modo ("overwrite") garante que os rankings e KPIs sejam recalculados do zero a cada execução.
df_gold_comercial.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medalhao.gold.fat_vendas_comercial")

In [0]:
# Importando as funções do Spark para os rankings de volume de vendas
from pyspark.sql import functions as F

# Agrupando por ID e Categoria para medir o nível de vendas de cada item
df_ranking_produtos = (
    df_base_vendas
    .groupBy("produtos.id_produto", "produtos.categoria_produto")
    .agg(F.count("itens.id_produto").alias("quantidade_vendida"))
)

display(df_ranking_produtos.orderBy(F.desc("quantidade_vendida")).limit(5))

display(df_ranking_produtos.filter(F.col("quantidade_vendida") > 0).orderBy(F.asc("quantidade_vendida")).limit(5))

In [0]:
# Importando as funções do Spark 
from pyspark.sql import functions as F

# Carregando as tabelas Silver para o cruzamento de dados de experiência do cliente
df_produtos = spark.table("medalhao.silver.dim_produtos")
df_vendedores = spark.table("medalhao.silver.dim_vendedores")
df_pedidos = spark.table("medalhao.silver.fat_pedido_total")
df_itens = spark.table("medalhao.silver.fat_itens_pedidos")
df_avaliacoes = spark.table("medalhao.silver.fat_avaliacoes_pedidos")

# Criando a Base de Satisfação: Unindo a avaliação do cliente com os dados do pedido e do vendedor, o inner join para analisar avaliações que tenham um pedido e vendedor correspondentes
df_base_satisfacao = (
    df_avaliacoes.alias("avaliacoes")
    .join(df_pedidos.alias("pedidos"), "id_pedido", "inner")
    .join(df_itens.alias("items"), "id_pedido", "inner")
    .join(df_produtos.alias("produtos"), "id_produto", "inner")
    .join(df_vendedores.alias("vendedores"), "id_vendedor", "inner")
)

# Construindo a Camada Gold de Satisfação 
df_gold_satisfacao = (
    df_base_satisfacao
    # Agrupando para identificar quais categorias e vendedores performam melhor no atendimento
    .groupBy("produtos.categoria_produto", "vendedores.nome_vendedor", "vendedores.estado")
    
    .agg(
        # Contagem total de feedbacks recebidos
        F.count("avaliacoes.nota_avaliacao").alias("total_avaliacoes"),
        
        # Média aritmética das notas para o Score de satisfação 
        F.round(F.avg("avaliacoes.nota_avaliacao"), 2).alias("avaliacao_media"),
        
        # Lógica Condicional: Notas 4 e 5 são consideradas positivas
        F.sum(F.when(F.col("avaliacoes.nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        
        # Lógica Condicional: Notas 1 e 2 são consideradas negativas
        F.sum(F.when(F.col("avaliacoes.nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
    )
    
    # Cálculo do Percentual de Satisfação
    .withColumn("percentual_satisfacao", 
                F.round((F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100, 2))
)
# Usamos o overwriteSchema para que o Job não quebre caso a estrutura da Silver mude, o modo ("overwrite") garante que os rankings e KPIs sejam recalculados do zero a cada execução.
df_gold_satisfacao.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medalhao.gold.fat_avaliacoes_clientes")

In [0]:
# Importando as funções do Spark 
from pyspark.sql import functions as F

# Agrupando por ID e Nome para identificar a reputação de cada item no catálogo
df_rank_produtos = df_base_satisfacao.groupBy("produtos.id_produto", "produtos.nome_produto").agg(
    # Média das notas para medir a qualidade percebida
    F.avg("avaliacoes.nota_avaliacao").alias("nota"),
    # Volume de avaliações para servir como critério de desempate
    F.count("avaliacoes.nota_avaliacao").alias("volume")
)

# Agrupando por ID e Nome do Vendedor para medir a performance do atendimento
df_rank_vendedores = df_base_satisfacao.groupBy("vendedores.id_vendedor", "vendedores.nome_vendedor").agg(
    F.avg("avaliacoes.nota_avaliacao").alias("nota"),
    F.count("avaliacoes.nota_avaliacao").alias("volume")
)

# Display para exibir o produto melhor avaliado
display(df_rank_produtos.orderBy(F.desc("nota"), F.desc("volume")).limit(1))

#Display para exibir o produto pior avaliado
display(df_rank_produtos.orderBy(F.asc("nota"), F.desc("volume")).limit(1))

#Display para exibir o vendedor melhor avaliado
display(df_rank_vendedores.orderBy(F.desc("nota"), F.desc("volume")).limit(1))

# Display para exibir o vendedor pior avaliado
display(df_rank_vendedores.orderBy(F.asc("nota"), F.desc("volume")).limit(1))